In [ ]:
%load_ext autoreload
%autoreload 2

# Run this cell first to prevent cached modules!

# EKB Orchestrator Verification

This notebook verifies the `KBIngestionPipeline` orchestrator, which sequences `ClassificationPipeline` and `RAGIngestion`.

In [ ]:
import sys
import os
sys.path.append("../..")
from google.cloud import bigquery
from pipelines.enterprise_knowledge_base.orchestrator import KBIngestionPipeline

In [ ]:
# Initialize the pipeline orchestrator
PROJECT_ID = "ag-core-dev-fdx7"
os.environ["PROJECT_ID"] = PROJECT_ID
pipeline = KBIngestionPipeline(project_id=PROJECT_ID)


In [ ]:
# Run the full pipeline (Classification + RAG Ingestion)
# Note: This will move the file from landing to domain-specific bucket AND index it.
GCS_URI = "gs://ag-core-dev-fdx7-kb-landing-zone/ingested/test_doc_for_embeddings.pdf"

try:
    pipeline.trigger_pipeline(GCS_URI)
    print("Pipeline execution triggered successfully.")
except Exception as e:
    print(f'Pipeline execution result: {e}')

In [ ]:
# Verify BigQuery: Check if chunks were generated and vectorized
bq_client = bigquery.Client(project=PROJECT_ID)
query = f"""
SELECT chunk_id, gcs_uri, filename, page_number, ARRAY_LENGTH(embedding) as embedding_length 
FROM `{PROJECT_ID}.knowledge_base.documents_chunks` 
WHERE filename = 'test_doc_for_embeddings.pdf'
LIMIT 5
"""

print("Checking BigQuery chunks...")
results = bq_client.query(query).result()
for row in results:
     print(dict(row))

## Domain Bucket Integrity Verification

In [ ]:
from google.cloud import storage
storage_client = storage.Client()

# We expect the file to still exist in the domain bucket path created by classification
# Note: You'll need to know the domain/tier assigned by classification
DOMAIN = "it" # Example
TIER = "internal-use-only" # Example
PROJECT = "unknown"
USER = "unknown"
FILENAME = "test_doc_for_embeddings.pdf"

DOMAIN_PATH = f"{TIER}/{PROJECT}/{USER}/{FILENAME}"
DOMAIN_BUCKET = f"kb-{DOMAIN}"

print(f"Checking domain bucket: {DOMAIN_BUCKET} for path {DOMAIN_PATH}...")
blob = storage_client.bucket(DOMAIN_BUCKET).get_blob(DOMAIN_PATH)
if blob:
    print(f"SUCCESS: Original file is UNTOUCHED in domain bucket: {blob.name}")
else:
    print("FAILURE: Original file was moved or deleted from domain bucket!")